# Liquidation task — data EDA

Explore `liquidation_task.tar`: extract Parquet files, validate schemas, and check data quality before building the liquidation signal.

**Data (3 months):** Binance trades, BBO, liquidations + Bybit liquidations for `btcusdt` / `ethusdt`.

- `timestamp`: int64, **microseconds** since Unix epoch (UTC)
- Train: `2025-12-01` → `2026-01-31` | Val: `2026-02-01` → `2026-02-28`
- Bybit events: shift **+200 ms** when aligning with Binance

In [1]:
from pathlib import Path
import tarfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path(".").resolve()
TAR_PATH = PROJECT_ROOT / "liquidation_task.tar"
DATA_ROOT = PROJECT_ROOT / "liquidation_task" / "data"

TRAIN_END = pd.Timestamp("2026-01-31", tz="UTC")
VAL_START = pd.Timestamp("2026-02-01", tz="UTC")
BYBIT_LAG_US = 200_000  # 200 ms in microseconds

EXPECTED_SCHEMAS = {
    "trades": ["timestamp", "ticker", "side", "price", "amount"],
    "bbo": ["timestamp", "ticker", "bid_price", "bid_amount", "ask_price", "ask_amount"],
    "liq": ["timestamp", "ticker", "side", "price", "amount"],
}

FILE_MAP = {
    "binance_trades": {
        "btc": DATA_ROOT / "binance_trades/perp_btcusdt.parquet",
        "eth": DATA_ROOT / "binance_trades/perp_ethusdt.parquet",
    },
    "binance_bbo": {
        "btc": DATA_ROOT / "binance_booktickers/perp_btcusdt.parquet",
        "eth": DATA_ROOT / "binance_booktickers/perp_ethusdt.parquet",
    },
    "binance_liquidations": {
        "btc": DATA_ROOT / "binance_liquidations/perp_btcusdt.parquet",
        "eth": DATA_ROOT / "binance_liquidations/perp_ethusdt.parquet",
    },
    "bybit_liquidations": {
        "btc": DATA_ROOT / "bybit_liquidations/btcusdt.parquet",
        "eth": DATA_ROOT / "bybit_liquidations/ethusdt.parquet",
    },
}

In [2]:
def extract_tar_if_needed(tar_path: Path = TAR_PATH, out_dir: Path = PROJECT_ROOT / "liquidation_task") -> None:
    """Extract archive once. Skips if data/ already exists."""
    marker = out_dir / "data" / "binance_trades" / "perp_btcusdt.parquet"
    if marker.exists():
        print(f"Already extracted: {marker}")
        return
    if not tar_path.exists():
        raise FileNotFoundError(f"Missing archive: {tar_path}")
    print(f"Extracting {tar_path.name} (~5.5 GB, may take a few minutes)...")
    extract_kwargs = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
    with tarfile.open(tar_path, "r") as tf:
        tf.extractall(path=PROJECT_ROOT, **extract_kwargs)
    print(f"Done. Data at: {out_dir / 'data'}")


extract_tar_if_needed()

Extracting liquidation_task.tar (~5.5 GB, may take a few minutes)...
Done. Data at: /Users/vitaliiaioffe/PycharmProjects/hft/liquidation_task/data


In [6]:
def parquet_metadata(path: Path) -> dict:
    pf = pq.ParquetFile(path)
    return {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "rows": pf.metadata.num_rows,
        "columns": pf.schema_arrow.names,
        "size_mb": path.stat().st_size / 1e6,
    }


def load_parquet(path: Path, columns=None) -> pd.DataFrame:
    return pd.read_parquet(path, columns=columns)


def iter_parquet_chunks(path: Path, batch_size: int = 5_000_000):
  """Yield pandas chunks from large parquet files."""
  pf = pq.ParquetFile(path)
  for batch in pf.iter_batches(batch_size=batch_size):
    yield batch.to_pandas()


def ts_to_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, unit="us", utc=True)


def audit_dataframe(name: str, df: pd.DataFrame, *, kind: str, is_bbo: bool = False) -> pd.DataFrame:
    """Run structure + anomaly checks; returns summary row."""
    row = {"name": name, "rows": len(df), "kind": kind}

    if list(df.columns) != EXPECTED_SCHEMAS[kind]:
        row["schema_ok"] = False
        row["columns"] = str(list(df.columns))
    else:
        row["schema_ok"] = True

    ts = df["timestamp"]
    dt = ts_to_dt(ts)
    row["t_min"] = dt.min()
    row["t_max"] = dt.max()
    row["monotonic"] = bool(ts.is_monotonic_increasing)
    row["dup_ts"] = int(ts.duplicated().sum())
    row["dup_ts_pct"] = 100.0 * row["dup_ts"] / max(len(df), 1)
    row["nulls"] = int(df.isna().sum().sum())

    if not is_bbo and "side" in df.columns:
        row["invalid_side"] = int((~df["side"].isin(["buy", "sell"])).sum())
        row["side_buy_pct"] = 100.0 * (df["side"] == "buy").mean()

    if is_bbo:
        row["crossed_book"] = int((df["bid_price"] > df["ask_price"]).sum())
        spread_bps = (df["ask_price"] - df["bid_price"]) / df["bid_price"] * 1e4
        row["spread_bps_median"] = float(spread_bps.median())
    elif {"price", "amount"}.issubset(df.columns):
        notional = df["price"] * df["amount"]
        row["notional_median"] = float(notional.median())
        row["notional_p99"] = float(notional.quantile(0.99))

    if "price" in df.columns and not is_bbo:
        row["non_positive_price"] = int((df["price"] <= 0).sum())
    if "amount" in df.columns:
        row["non_positive_amount"] = int((df["amount"] <= 0).sum())

    row["tickers"] = ", ".join(sorted(df["ticker"].astype(str).unique()))
    return pd.DataFrame([row])

## File inventory

In [7]:
inventory = []
for group, paths in FILE_MAP.items():
    kind = "bbo" if "bbo" in group or "booktickers" in group else ("liq" if "liquidation" in group else "trades")
    for sym, path in paths.items():
        if path.exists():
            meta = parquet_metadata(path)
            meta["group"] = group
            meta["symbol"] = sym
            meta["kind"] = kind
            inventory.append(meta)

inventory_df = pd.DataFrame(inventory).sort_values(["group", "symbol"])
inventory_df

,path,rows,columns,size_mb,group,symbol,kind
2,liquidation_task/data/binance_booktickers/perp...,99169477,"[timestamp, ticker, bid_price, bid_amount, ask...",796.614560,binance_bbo,btc,bbo
3,liquidation_task/data/binance_booktickers/perp...,107797036,"[timestamp, ticker, bid_price, bid_amount, ask...",1249.317902,binance_bbo,eth,bbo
4,liquidation_task/data/binance_liquidations/per...,114255,"[timestamp, ticker, side, price, amount]",1.445862,binance_liquidations,btc,liq
5,liquidation_task/data/binance_liquidations/per...,131769,"[timestamp, ticker, side, price, amount]",1.812943,binance_liquidations,eth,liq
0,liquidation_task/data/binance_trades/perp_btcu...,401902513,"[timestamp, ticker, side, price, amount]",1294.425892,binance_trades,btc,trades
1,liquidation_task/data/binance_trades/perp_ethu...,705880385,"[timestamp, ticker, side, price, amount]",2118.505612,binance_trades,eth,trades
6,liquidation_task/data/bybit_liquidations/btcus...,228655,"[timestamp, ticker, side, price, amount]",2.437223,bybit_liquidations,btc,liq
7,liquidation_task/data/bybit_liquidations/ethus...,160214,"[timestamp, ticker, side, price, amount]",1.798941,bybit_liquidations,eth,liq


## Liquidations (full load)

Small enough to load entirely into memory.

In [8]:
liq_binance_btc = load_parquet(FILE_MAP["binance_liquidations"]["btc"])
liq_binance_eth = load_parquet(FILE_MAP["binance_liquidations"]["eth"])
liq_bybit_btc = load_parquet(FILE_MAP["bybit_liquidations"]["btc"])
liq_bybit_eth = load_parquet(FILE_MAP["bybit_liquidations"]["eth"])

display(liq_binance_btc.head(3))
display(liq_bybit_btc.head(3))

,timestamp,ticker,side,price,amount
0,1764547206091000,perp:btcusdt,sell,89936.0,0.014
1,1764547207101000,perp:btcusdt,sell,89930.2,0.004
2,1764547209066000,perp:btcusdt,sell,89924.7,0.048


,timestamp,ticker,side,price,amount
0,1764547202915000,btcusdt,sell,89838.9,0.032
1,1764547205124000,btcusdt,sell,89826.9,0.004
2,1764547207236000,btcusdt,sell,89821.6,0.005


In [9]:
liq_reports = pd.concat(
    [
        audit_dataframe("binance_liq_btc", liq_binance_btc, kind="liq"),
        audit_dataframe("binance_liq_eth", liq_binance_eth, kind="liq"),
        audit_dataframe("bybit_liq_btc", liq_bybit_btc, kind="liq"),
        audit_dataframe("bybit_liq_eth", liq_bybit_eth, kind="liq"),
    ],
    ignore_index=True,
)
liq_reports

,name,rows,kind,schema_ok,t_min,t_max,monotonic,dup_ts,dup_ts_pct,nulls,invalid_side,side_buy_pct,notional_median,notional_p99,non_positive_price,non_positive_amount,tickers
0,binance_liq_btc,114255,liq,True,2025-12-01 00:00:06.091000+00:00,2026-02-28 23:59:10.548000+00:00,True,0,0.000000,0,0,43.802022,1115.85760,163372.383900,0,0,perp:btcusdt
1,binance_liq_eth,131769,liq,True,2025-12-01 00:00:09.455000+00:00,2026-02-28 23:49:50.541000+00:00,True,0,0.000000,0,0,44.240299,727.96146,148877.707766,0,0,perp:ethusdt
2,bybit_liq_btc,228655,liq,True,2025-12-01 00:00:02.915000+00:00,2026-02-28 23:58:42.271000+00:00,False,5094,2.227810,0,0,32.737968,1118.23920,200391.538194,0,0,btcusdt
3,bybit_liq_eth,160214,liq,True,2025-12-01 00:00:07.779000+00:00,2026-02-28 23:34:22.075000+00:00,False,2664,1.662776,0,0,29.677182,911.98585,179374.750596,0,0,ethusdt


In [13]:
# Bybit: apply +200 ms lag for cross-exchange alignment (per task spec)
for label, df in [("bybit_btc", liq_bybit_btc), ("bybit_eth", liq_bybit_eth)]:
    adj = df.copy()
    adj["timestamp_adj"] = adj["timestamp"] + BYBIT_LAG_US
    print(f"{label}: monotonic before={df['timestamp'].is_monotonic_increasing}, "
          f"after lag={adj['timestamp_adj'].is_monotonic_increasing}")

bybit_btc: monotonic before=False, after lag=False
bybit_eth: monotonic before=False, after lag=False


## Train / validation split (liquidations)

In [14]:
def split_counts(df: pd.DataFrame, label: str) -> pd.DataFrame:
    dt = ts_to_dt(df["timestamp"])
    train = (dt <= TRAIN_END).sum()
    val = (dt >= VAL_START).sum()
    gap = len(df) - train - val
    return pd.DataFrame(
        [{"dataset": label, "train": train, "val": val, "other": gap, "total": len(df)}]
    )


split_summary = pd.concat(
    [
        split_counts(liq_binance_btc, "binance_liq_btc"),
        split_counts(liq_binance_eth, "binance_liq_eth"),
        split_counts(liq_bybit_btc, "bybit_liq_btc"),
        split_counts(liq_bybit_eth, "bybit_liq_eth"),
    ],
    ignore_index=True,
)
split_summary

,dataset,train,val,other,total
0,binance_liq_btc,61194,50595,2466,114255
1,binance_liq_eth,73578,55135,3056,131769
2,bybit_liq_btc,110166,104190,14299,228655
3,bybit_liq_eth,83084,63098,14032,160214


## Trades & BBO (chunked sample)

Full tables are 100M–700M rows — audit the **first N rows** here. For full-file stats use chunked iteration or DuckDB.

In [15]:
SAMPLE_ROWS = 2_000_000  # increase if you have more RAM; decrease for laptops

large_reports = []
for sym in ["btc", "eth"]:
    trades_path = FILE_MAP["binance_trades"][sym]
    bbo_path = FILE_MAP["binance_bbo"][sym]

    trades_sample = pq.read_table(trades_path).slice(0, SAMPLE_ROWS).to_pandas()
    bbo_sample = pq.read_table(bbo_path).slice(0, SAMPLE_ROWS).to_pandas()

    large_reports.append(audit_dataframe(f"trades_{sym} (first {SAMPLE_ROWS:,})", trades_sample, kind="trades"))
    large_reports.append(audit_dataframe(f"bbo_{sym} (first {SAMPLE_ROWS:,})", bbo_sample, kind="bbo", is_bbo=True))

    print(f"\n--- trades_{sym} head ---")
    display(trades_sample.head(3))
    print(f"duplicate timestamp groups (sample): {trades_sample.groupby('timestamp').size().max()} max trades per µs")

pd.concat(large_reports, ignore_index=True)


--- trades_btc head ---


,timestamp,ticker,side,price,amount
0,1764547200047000,perp:btcusdt,sell,90320.5,0.003
1,1764547200047000,perp:btcusdt,sell,90320.5,0.003
2,1764547200047000,perp:btcusdt,sell,90320.5,0.003


duplicate timestamp groups (sample): 455 max trades per µs

--- trades_eth head ---


,timestamp,ticker,side,price,amount
0,1764547200016000,perp:ethusdt,buy,2990.02,0.200
1,1764547200025000,perp:ethusdt,sell,2990.01,0.053
2,1764547200036000,perp:ethusdt,sell,2990.01,2.151


duplicate timestamp groups (sample): 496 max trades per µs


,name,rows,kind,schema_ok,t_min,t_max,monotonic,dup_ts,dup_ts_pct,nulls,invalid_side,side_buy_pct,notional_median,notional_p99,non_positive_price,non_positive_amount,tickers,crossed_book,spread_bps_median
0,"trades_btc (first 2,000,000)",2000000,trades,True,2025-12-01 00:00:00.047000+00:00,2025-12-01 04:25:25.827000+00:00,True,1542019,77.10095,0,0.0,48.95565,262.585800,62534.731844,0.0,0.0,perp:btcusdt,NaN,NaN
1,"bbo_btc (first 2,000,000)",2000000,bbo,True,2025-12-01 00:00:02.075000+00:00,2025-12-02 14:30:03.759000+00:00,True,0,0.00000,0,NaN,NaN,NaN,NaN,NaN,NaN,perp:btcusdt,0.0,0.011566
2,"trades_eth (first 2,000,000)",2000000,trades,True,2025-12-01 00:00:00.016000+00:00,2025-12-01 01:31:12.987000+00:00,True,1663692,83.18460,0,0.0,47.20190,55.120235,28933.450451,0.0,0.0,perp:ethusdt,NaN,NaN
3,"bbo_eth (first 2,000,000)",2000000,bbo,True,2025-12-01 00:00:00.683000+00:00,2025-12-02 10:24:47.267000+00:00,True,0,0.00000,0,NaN,NaN,NaN,NaN,NaN,NaN,perp:ethusdt,0.0,0.035628


## Optional: DuckDB global stats (no full DataFrame)

Uncomment if `duckdb` is installed. Scans entire parquet without loading into pandas.

In [ ]:
# import duckdb
#
# con = duckdb.connect()
# path = str(FILE_MAP["binance_trades"]["btc"])
# con.execute(f"""
#     SELECT
#         COUNT(*) AS n,
#         MIN(timestamp) AS ts_min,
#         MAX(timestamp) AS ts_max,
#         COUNT(DISTINCT timestamp) AS n_distinct_ts
#     FROM read_parquet('{path}')
# """).df()

## Summary checklist

| Check | Expected |
|-------|----------|
| Schema columns | Match `EXPECTED_SCHEMAS` |
| `timestamp` | int64 µs UTC; covers Dec 2025 – Feb 2026 |
| `side` | only `buy` / `sell` |
| Binance `ticker` | `perp:btcusdt`, `perp:ethusdt` |
| Bybit `ticker` | `btcusdt`, `ethusdt` |
| BBO | `bid_price` ≤ `ask_price` |
| Trades | many rows per timestamp is normal |
| Bybit liquidations | may have duplicate timestamps → sort / dedupe; use **+200 ms** lag |